# DS Automation Assignment

# Task 1
Using our prepared churn data from week 2:
- use pycaret to find an ML algorithm that performs best on the data
    - Choose a metric you think is best to use for finding the best model; by default, it is accuracy but it could be AUC, precision, recall, etc. The week 3 FTE has some information on these different metrics.
- save the model to disk
- create a Python script/file/module with a function that takes a pandas dataframe as an input and returns the probability of churn for each row in the dataframe
    - your Python file/function should print out the predictions for new data (new_churn_data.csv)
    - the true values for the new data are [1, 0, 0, 1, 0] if you're interested
- test your Python module and function with the new data, new_churn_data.csv
- write a short summary of the process and results at the end of this notebook
- upload this Jupyter Notebook and Python file to a Github repository, and turn in a link to the repository in the week 5 assignment dropbox

Additional challenges:
- return the probability of churn for each new prediction, and the percentile where that prediction is in the distribution of probability predictions from the training dataset (e.g. a high probability of churn like 0.78 might be at the 90th percentile)
- use other autoML packages, such as TPOT, H2O, MLBox, etc, and compare performance and features with pycaret
- create a class in your Python module to hold the functions that you created
- accept user input to specify a file using a tool such as Python's `input()` function, the `click` package for command-line arguments, or a GUI
- Use the unmodified churn data (new_unmodified_churn_data.csv) in your Python script. This will require adding the same preprocessing steps from week 2 since this data is like the original unmodified dataset from week 1.


# Task 2: Critical Reflection
What are your observations based on the analyses you had performed? Is there one model that seems particularly useful for predicting churn. Why or why not?

## Task 1: Churn Prediction with AutoML

### Environment Check

In [1]:
import sys
import pycaret

print("Python version:", sys.version)
print("PyCaret version:", pycaret.__version__)

Python version: 3.10.14 | packaged by Anaconda, Inc. | (main, May  6 2024, 19:44:50) [MSC v.1916 64 bit (AMD64)]
PyCaret version: 3.3.2


## Load and Inspect the Prepared Churn Data

In [2]:
import pandas as pd

df = pd.read_csv('churn_data_cleaned.csv')

df.head()

,tenure,PhoneService,MonthlyCharges,TotalCharges,Churn,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,ChargesPerMonth
0,1,0,29.85,29.85,0,0,0,0,1,0,29.850000
1,34,1,56.95,1889.50,0,1,0,0,0,1,55.573529
2,2,1,53.85,108.15,1,0,0,0,0,1,54.075000
3,45,0,42.30,1840.75,0,1,0,0,0,0,40.905556
4,2,1,70.70,151.65,1,0,0,0,1,0,75.825000


In [3]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nChurn counts:")
print(df['Churn'].value_counts())

Dataset shape: (7043, 11)

Column names:
['tenure', 'PhoneService', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'ChargesPerMonth']

Data types:
tenure                                     int64
PhoneService                               int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
Contract_One year                          int64
Contract_Two year                          int64
PaymentMethod_Credit card (automatic)      int64
PaymentMethod_Electronic check             int64
PaymentMethod_Mailed check                 int64
ChargesPerMonth                          float64
dtype: object

Missing values:
tenure                                   0
PhoneService                             0
MonthlyCharges                           0
TotalCharge

### Model Evaluation Metric

The dataset contains 7,043 customers. There are 5,174 non-churn customers and 1,869 churn customers, so the classes are somewhat imbalanced. I will use AUC as the main metric for comparing models because AUC measures how well a model separates churners from non-churners across different classification thresholds. I will also review accuracy, precision, recall, and F1 score when evaluating the models.

## PyCaret Model Setup

In [4]:
from pycaret.classification import setup, compare_models, predict_model, save_model, load_model

automl = setup(
    data=df,
    target='Churn',
    session_id=123
)

,Description,Value
0,Session id,123
1,Target,Churn
2,Target type,Binary
3,Original data shape,"(7043, 11)"
4,Transformed data shape,"(7043, 11)"
5,Transformed train set shape,"(4930, 11)"
6,Transformed test set shape,"(2113, 11)"
7,Numeric features,10
8,Preprocess,True
9,Imputation type,simple


## Compare Machine Learning Models

In [5]:
best_model = compare_models(sort='AUC')

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.7923,0.8367,0.4963,0.6410,0.5574,0.4251,0.4321,0.1740
ada,Ada Boost Classifier,0.7899,0.8349,0.5054,0.6296,0.5596,0.4241,0.4291,0.0750
lr,Logistic Regression,0.7945,0.8348,0.5169,0.6409,0.5714,0.4385,0.4434,0.8470
ridge,Ridge Classifier,0.7876,0.8246,0.4519,0.6428,0.5298,0.3982,0.4090,0.0110
lda,Linear Discriminant Analysis,0.7890,0.8246,0.5268,0.6214,0.5695,0.4312,0.4342,0.0130
qda,Quadratic Discriminant Analysis,0.7174,0.8209,0.7829,0.4805,0.5952,0.3969,0.4252,0.0120
nb,Naive Bayes,0.7302,0.8134,0.7585,0.4949,0.5986,0.4088,0.4304,0.0130
rf,Random Forest Classifier,0.7694,0.7959,0.4634,0.5831,0.5152,0.3668,0.3716,0.1630
et,Extra Trees Classifier,0.7564,0.7745,0.4695,0.5487,0.5050,0.3450,0.3474,0.1280
knn,K Neighbors Classifier,0.7677,0.7532,0.4427,0.5815,0.5019,0.3544,0.3603,0.5530


## Best Model and Holdout Performance

In [6]:
holdout_predictions = predict_model(best_model)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.7960,0.8425,0.4955,0.6526,0.5633,0.4335,0.4405


## Save the Best Model

In [7]:
save_model(best_model, 'churn_model')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['tenure', 'PhoneService',
                                              'MonthlyCharges', 'TotalCharges',
                                              'Contract_One year',
                                              'Contract_Two year',
                                              'PaymentMethod_Credit card '
                                              '(automatic)',
                                              'PaymentMethod_Electronic check',
                                              'PaymentMethod_Mailed check',
                                              'ChargesPerMonth'],
                                     transformer=SimpleImputer(add_...
                                             criterion='friedman_mse', init=None,
                                             learning_rate=0.1, loss='log

## Test the Prediction Module

### Load New Customer Data

In [8]:
new_data = pd.read_csv('new_churn_data.csv')

new_data

,customerID,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure
0,9305-CKSKC,22,1,0,2,97.40,811.70,36.895455
1,1452-KNGVK,8,0,1,1,77.30,1701.95,212.743750
2,6723-OKKJM,28,1,0,0,28.25,250.90,8.960714
3,7832-POPKP,62,1,0,2,101.70,3106.56,50.105806
4,6348-TACGU,10,0,0,1,51.15,3440.97,344.097000


### Load Unmodified Customer Data

In [9]:
raw_new = pd.read_csv('new_unmodified_churn_data.csv')

raw_new[['customerID', 'PhoneService', 'Contract', 'PaymentMethod',
         'MonthlyCharges', 'TotalCharges']]

,customerID,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges
0,9305-CKSKC,Yes,Month-to-month,Electronic check,97.40,811.70
1,1452-KNGVK,No,One year,Mailed check,77.30,1701.95
2,6723-OKKJM,Yes,Month-to-month,Credit card (automatic),28.25,250.90
3,7832-POPKP,Yes,Month-to-month,Electronic check,101.70,3106.56
4,6348-TACGU,No,Two year,Credit card (automatic),51.15,3440.97


### Test Predictions with Unmodified Customer Data

### Generate Predictions with the Python Module

In [10]:
import importlib
import churn_predictor

importlib.reload(churn_predictor)

final_new_results = churn_predictor.make_predictions(new_data)

final_new_results

Transformation Pipeline and Model Successfully Loaded
Predictions:
   churn_probability  predicted_churn
0           0.590917                1
1           0.207168                0
2           0.165786                0
3           0.440878                0
4           0.317632                0


,churn_probability,predicted_churn
0,0.590917,1
1,0.207168,0
2,0.165786,0
3,0.440878,0
4,0.317632,0


In [11]:
raw_results = churn_predictor.make_predictions(raw_new)

raw_results

Transformation Pipeline and Model Successfully Loaded
Predictions:
   churn_probability  predicted_churn
0           0.590917                1
1           0.207168                0
2           0.165786                0
3           0.440878                0
4           0.073300                0


,churn_probability,predicted_churn
0,0.590917,1
1,0.207168,0
2,0.165786,0
3,0.440878,0
4,0.073300,0


The two supplied new-customer files contain different category values for customer 6348-TACGU under the mappings used in my script. The prepared file represents this customer as having a month-to-month contract and paying by mailed check, while the unmodified file lists a two-year contract and credit card payment. The predicted churn probabilities are therefore different: approximately 0.3176 and 0.0733, respectively. I retained both input files unchanged and reported the results separately.

In [24]:
%run churn_predictor.py

Task was destroyed but it is pending!
task: <Task pending name='Task-144' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\Users\wajid\anaconda3\envs\week5-pycaret\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-145' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\wajid\anaconda3\envs\week5-pycaret\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\wajid\anaconda3\envs\week5-pycaret\lib\site-packages\zmq\eventloop\zmqstream.py:564]>
Task was destroyed but it is pending!
task: <Task pending name='Task-145' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\wajid\anaconda3\envs\week5-pycaret\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]>


Transformation Pipeline and Model Successfully Loaded


Enter the CSV filename:  new_churn_data.csv


Predictions:
   churn_probability  predicted_churn
0           0.590917                1
1           0.207168                0
2           0.165786                0
3           0.440878                0
4           0.317632                0


## Additional Challenge: Churn Probability Percentile

### Create the Training Probability Distribution

In [25]:
from pycaret.classification import get_config

training_features = get_config('X_train')

training_features.shape

(4930, 10)

In [26]:
training_probabilities = churn_predictor.predict_churn_probability(
    training_features
)

training_probabilities.describe()

Transformation Pipeline and Model Successfully Loaded


count    4930.000000
mean        0.265364
std         0.242072
min         0.005601
25%         0.049690
50%         0.184981
75%         0.446641
max         0.958654
Name: churn_probability, dtype: float64

### Test the ChurnPredictor Class

In [27]:
predictor = churn_predictor.ChurnPredictor()

class_results = predictor.predict(
    new_data,
    training_probabilities
)

class_results.round({
    'churn_probability': 4,
    'percentile': 1
})

Transformation Pipeline and Model Successfully Loaded
Predictions:
   churn_probability  predicted_churn  percentile
0           0.590917                1   86.430020
1           0.207168                0   52.576065
2           0.165786                0   47.647059
3           0.440878                0   74.401623
4           0.317632                0   64.523327


,churn_probability,predicted_churn,percentile
0,0.5909,1,86.4
1,0.2072,0,52.6
2,0.1658,0,47.6
3,0.4409,0,74.4
4,0.3176,0,64.5


## Additional Challenge: PyCaret vs. H2O AutoML

### Initialize H2O

In [15]:
import h2o

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,12 days 18 hours 10 mins
H2O_cluster_timezone:,America/Denver
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,1 month and 14 days
H2O_cluster_name:,H2O_from_python_wajid_crub8z
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.130 Gb
H2O_cluster_total_cores:,16
H2O_cluster_allowed_cores:,16
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+


### Use the Same PyCaret Training and Test Split

In [16]:
h2o_X_train = get_config('X_train')
h2o_y_train = get_config('y_train')

h2o_X_test = get_config('X_test')
h2o_y_test = get_config('y_test')

print("Training features:", h2o_X_train.shape)
print("Training target:", h2o_y_train.shape)

print("Test features:", h2o_X_test.shape)
print("Test target:", h2o_y_test.shape)

Training features: (4930, 10)
Training target: (4930,)
Test features: (2113, 10)
Test target: (2113,)


### Prepare Training and Test Data for H2O

In [17]:
h2o_train_pd = h2o_X_train.copy()
h2o_train_pd['Churn'] = h2o_y_train.values

h2o_test_pd = h2o_X_test.copy()
h2o_test_pd['Churn'] = h2o_y_test.values

print("H2O training data:", h2o_train_pd.shape)
print("H2O test data:", h2o_test_pd.shape)

print("\nTraining churn counts:")
print(h2o_train_pd['Churn'].value_counts())

H2O training data: (4930, 11)
H2O test data: (2113, 11)

Training churn counts:
Churn
0    3622
1    1308
Name: count, dtype: int64


### Convert the Data to H2O Frames

In [18]:
h2o_train = h2o.H2OFrame(h2o_train_pd)
h2o_test = h2o.H2OFrame(h2o_test_pd)

h2o_train['Churn'] = h2o_train['Churn'].asfactor()
h2o_test['Churn'] = h2o_test['Churn'].asfactor()

print("H2O training shape:", h2o_train.shape)
print("H2O test shape:", h2o_test.shape)
print("Churn type:", h2o_train.types['Churn'])

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
H2O training shape: (4930, 11)
H2O test shape: (2113, 11)
Churn type: enum


### Define Predictors and Target

In [19]:
from h2o.automl import H2OAutoML

x = [col for col in h2o_train.columns if col != 'Churn']
y = 'Churn'

print("Predictor columns:", len(x))
print("Target column:", y)

Predictor columns: 10
Target column: Churn


### Train H2O AutoML Models

In [20]:
aml = H2OAutoML(
    max_models=10,
    seed=123,
    sort_metric='AUC'
)

aml.train(
    x=x,
    y=y,
    training_frame=h2o_train
)

AutoML progress: |█
20:03:28.174: AutoML: XGBoost is not available; skipping it.

██████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),4/5
# GBM base models (used / total),1/1
# GLM base models (used / total),1/1
# DeepLearning base models (used / total),1/1
# DRF base models (used / total),1/2
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


### Review the H2O AutoML Leaderboard

In [21]:
h2o_leaderboard = aml.leaderboard

h2o_leaderboard.head(rows=10)

model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
StackedEnsemble_BestOfFamily_1_AutoML_2_20260925_200328,0.839357,0.422087,0.641574,0.248379,0.371579,0.138071
StackedEnsemble_AllModels_1_AutoML_2_20260925_200328,0.838977,0.422418,0.640882,0.246437,0.37181,0.138243
GBM_1_AutoML_2_20260925_200328,0.837369,0.424334,0.640022,0.248093,0.372489,0.138748
GLM_1_AutoML_2_20260925_200328,0.833671,0.429221,0.617605,0.245068,0.375441,0.140956
GBM_5_AutoML_2_20260925_200328,0.829564,0.432945,0.623129,0.254199,0.376989,0.142121
GBM_2_AutoML_2_20260925_200328,0.826913,0.436132,0.623417,0.251948,0.378319,0.143125
DeepLearning_1_AutoML_2_20260925_200328,0.826482,0.439061,0.602026,0.247882,0.379425,0.143963
GBM_3_AutoML_2_20260925_200328,0.826123,0.437254,0.623113,0.260793,0.378928,0.143587
GBM_grid_1_AutoML_2_20260925_200328_model_1,0.825498,0.437444,0.623614,0.255134,0.378926,0.143585
GBM_4_AutoML_2_20260925_200328,0.820991,0.444225,0.614481,0.262843,0.382098,0.145999


### Evaluate the H2O Leader on the Test Data

In [22]:
h2o_leader = aml.leader

h2o_test_performance = h2o_leader.model_performance(
    test_data=h2o_test
)

print("H2O test AUC:", h2o_test_performance.auc())

H2O test AUC: 0.8431091157175148


### Compare PyCaret and H2O Test Performance

In [23]:
comparison = pd.DataFrame({
    'AutoML Tool': ['PyCaret', 'H2O AutoML'],
    'Selected Model': [
        'Gradient Boosting Classifier',
        'Stacked Ensemble (Best of Family)'
    ],
    'Test AUC': [
        0.8425,
        h2o_test_performance.auc()
    ]
})

comparison.round({
    'Test AUC': 4
})

,AutoML Tool,Selected Model,Test AUC
0,PyCaret,Gradient Boosting Classifier,0.8425
1,H2O AutoML,Stacked Ensemble (Best of Family),0.8431


### Interpretation
I compared the best model selected by PyCaret with the best model selected by H2O AutoML using the same test data. The PyCaret Gradient Boosting Classifier achieved an AUC of 0.8425, while the H2O Stacked Ensemble achieved an AUC of 0.8431. H2O had a slightly higher AUC, but the difference was very small. This suggests that both AutoML tools produced very similar predictive performance for this churn dataset. PyCaret was easier for me to use and interpret, while H2O tested multiple model types and selected a stacked ensemble.

### Task 2: Critical Reflection

Based on my analysis, both PyCaret and H2O AutoML produced similar results for predicting customer churn. PyCaret selected a Gradient Boosting Classifier with a test AUC of 0.8425, while H2O selected a Stacked Ensemble with a test AUC of 0.8431. This difference was very small, so I would not consider one model clearly better based only on AUC.

The Gradient Boosting Classifier seems useful for ranking customers by their likelihood of churn. However, its holdout recall was 49.55% at the default classification threshold, meaning it identified only about half of the customers who actually churned. This is an important limitation because a company using the model for retention could miss many customers who need attention. Lowering the threshold might identify more churners, but it could also incorrectly flag more customers who would stay. This trade-off would need further evaluation using validation data and the costs of retention efforts.

I found PyCaret easier to use for this assignment, although ease of using the tool does not necessarily mean its model is easy to interpret. H2O selected a stacked ensemble that combines several models, but its slightly higher AUC did not show a meaningful practical advantage in this comparison. Overall, I would consider the Gradient Boosting Classifier a useful starting point, while reviewing recall, precision, and business needs before using it to guide retention decisions.

## Summary

In this assignment, I used PyCaret to compare several classification models for predicting customer churn. I selected AUC as the main evaluation metric because the churn classes were not evenly distributed and AUC measures how well a model separates churners from non-churners across different thresholds. PyCaret selected the Gradient Boosting Classifier as the best model, with a test AUC of 0.8425.

I saved the trained model and created a separate Python module to load the model, prepare new customer data, calculate churn probabilities, and make churn predictions. I also added a percentile calculation so that each customer's churn probability could be compared with the training population. As an additional challenge, I trained models using H2O AutoML on the same training and test data. H2O selected a Stacked Ensemble with a test AUC of 0.8431. Overall, PyCaret and H2O produced very similar predictive performance, and the assignment showed how an AutoML model can be saved and reused for predictions on new customer data.

### AI Use Note
I used ChatGPT to help me understand the assignment, develop and troubleshoot Python code, organize my notebook, and improve the wording of my interpretations and critical reflection. I also used it for guidance on GitHub and to review my notebook, prediction script, and data files. I ran the code in my own Jupyter environment and checked the outputs against the written results.

https://chatgpt.com/share/6ab73091-e404-83e8-ae25-0a8e16ecb12b